## 模块：文件操作——让数据持久化

**驱动问题**：在开发一个调用大模型API的应用时，我们需要读取配置文件（如API Key），并将生成的对话记录保存到日志文件中。程序运行时的数据都在内存里，一关就没了。文件操作能让我们把数据存到硬盘上，实现永久保存。本节聚焦最核心、最常用的文件读写技能。

---

### 1. 为什么需要文件操作？

- **数据持久化**：将内存中的数据存入硬盘（如保存生成的文本、用户配置），断电不丢失。
- **数据交换**：程序间通过文件传递数据（如读取 `.txt`, `.json`, `.csv` 文件）。

---

### 2. 文件操作核心：打开 → 读写 → 关闭

#### 2.1 打开文件 `open()` 与模式

| 模式  | 说明                 |
| ----- | -------------------- |
| `'r'` | **只读**（默认），文件必须存在 |
| `'w'` | **只写**，覆盖原内容，文件不存在则新建 |
| `'a'` | **只写**，追加到文件末尾 |
| `'rb'`| 二进制模式读取（图片、视频） |

#### 2.2 读取与写入

In [ ]:
# # 1. 读取全部内容
# with open('config.txt', 'r', encoding='utf-8') as f:
#     content = f.read()
#     print(content)

# # 2. 按行读取（处理文本文件最常用）
# with open('data.txt', 'r', encoding='utf-8') as f:
#     for line in f:  # 一行行读，不占内存
#         print(line.strip())

# 3. 写入文件
with open('output.txt', 'w', encoding='utf-8') as f:
    f.write('第一行内容\n')
    f.write('第二行内容\n')
    f.write("白泽书写\n")

#### 2.3 使用 `with` 语句（最佳实践）

**永远使用 `with open(...) as f:` 来处理文件。** 它保证了即使程序出错，文件也会被自动关闭，避免资源泄露。

In [ ]:
# ✅ 推荐：自动管理资源
with open('data.txt', 'r', encoding='utf-8') as f:
    content = f.read()
# 文件在这里已自动关闭

# ❌ 不推荐：容易忘记关闭文件，导致资源泄露
# f = open('data.txt', 'r')
# content = f.read()
# .....
# f.close()

---

### 3. 大文件处理：“化整为零”

AI 应用中经常遇到几个 GB 的日志或数据文件。**永远不要用 `f.read()` 一次性读取整个大文件！** 要用“流式处理”：

In [ ]:
# 处理超大日志文件，统计含有 "ERROR" 的行
error_count = 0
with open('huge_app.log', 'r', encoding='utf-8') as f:
    for line in f:  # 每次只读取一行到内存
        if 'ERROR' in line:
            error_count += 1
print(f"错误行数: {error_count}")

**核心思想**：把文件想象成水流，一行一行地处理，而不是一次性灌满整个水池。

In [ ]:
chunk_size = 1024
with open("dataset.txt", "r", encoding="utf-8") as f:
    while True:
        # 1. 每次只读取 chunk_size 大小的数据
        chunk = f.read(chunk_size)
        # 2. 如果读取结果为空，说明到了文件末尾
        if not chunk:
            break
        # 3. 在这里处理这一小块数据 (这里简化处理的逻辑)
        print(f"处理了 {len(chunk)} 个字符")

---

### 4. 路径操作基础（`os.path`）

为了让代码在不同操作系统（Windows/Linux/macOS）上都能运行，拼接路径时不要手动拼接字符串，要用 `os.path.join`。

In [ ]:
import os

# ✅ 正确：跨平台路径拼接
log_dir = "my_project"
log_file = "app.log"
full_path = os.path.join(log_dir, "logs", log_file)
print(full_path)  # Windows: my_project\logs\app.log  Linux/Mac: my_project/logs/app.log

# 获取当前文件所在目录（常用）
current_dir = os.path.dirname(os.path.abspath(__file__))
print(current_dir)


# 列出目录下所有文件
files = os.listdir('.')
print(files)

print("*"*100)
for root, dirs, files in os.walk(current_dir):
    print('当前目录: ', root)
    print('子目录列表: ', dirs)
    print('文件列表: ', files)

---

### 5. 实战：一个简易的日志分析器

**需求**：读取当前目录下的 `huge_app.log` 文件，筛选出包含 “ERROR” 的行，并写入到 `error_report.txt` 中。

In [9]:
import os

input_file = 'huge_app.log'
output_file = 'error_report.txt'

# 检查文件是否存在
if not os.path.exists(input_file):
    print(f"错误：文件 {input_file} 不存在！")
else:
    error_lines = []
    # 1. 流式读取源文件
    with open(input_file, 'r', encoding='utf-8') as f_in:
        for line in f_in:
            if 'ERROR' in line:
                error_lines.append(line)
    
    # 2. 写入报告文件
    if error_lines:
        with open(output_file, 'w', encoding='utf-8') as f_out:
            f_out.writelines(error_lines)
        print(f"分析完成，错误报告已保存至 {output_file}")
    else:
        print("未发现任何错误。")

分析完成，错误报告已保存至 error_report.txt


---

### 6. 避坑指南

1.  **编码问题**：读取文本文件时，务必指定 `encoding='utf-8'`，否则读取中文时容易报错。
2.  **资源泄露**：始终坚持用 `with open()` 来处理文件。
3.  **大文件**：不要对未知大小的文件直接使用 `.read()` 或 `.readlines()`，优先用 `for line in f` 逐行处理 或者 chunk_size逐块处理。